Fit and save a global propensity score model

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str((Path.cwd().parent / "src").resolve()))

import pandas as pd
import ee
import json
import geemap
import numpy as np
from pyproj import Transformer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

from utils.variables import (
    BIOME_ASSET_ID,
)

from psm.tiling import tiles_to_feature_collection
from utils.variables import PROJECT, PAS_ASSET_ID, OECMS_ASSET_ID, EE_CRS_METERS, PSM_CELL_SIZE
from psm.tiling import build_tile_grid, filter_tiles_to_land
from psm.allocation import (
    compute_global_allocation,
    compute_tile_pixel_counts,
    compute_tile_allocations,
    validate_allocation,
    save_allocation,
    load_allocation,
)
from psm.sampling import build_sample_export_task, TileTaskManager

ee.Initialize(project=PROJECT)

# Configuration
GCS_BUCKET = "tpae"
GCS_PREFIX = "psm_samples/"
STATE_FILE = Path("data/psm_tile_state.json")
STATE_FILE.parent.mkdir(parents=True, exist_ok=True)

TOTAL_POINTS = 100000
TREAT_CONTROL = (1, 2)
MIN_PER_STRATUM = 50

EE_CRS_1km = ee.Projection(EE_CRS_METERS).atScale(PSM_CELL_SIZE)

In [ ]:
# Make a binary PA mask (protected = 1, unprotected = 0)

PAs = ee.FeatureCollection(PAS_ASSET_ID)
OECMS = ee.FeatureCollection(OECMS_ASSET_ID)
all_PAs = (
    ee.FeatureCollection([PAs, OECMS])
    .flatten()
    .filter(ee.Filter.eq("REALM", "Terrestrial"))
)
protected_mask = (
    ee.Image.constant(0)
    .rename("protected")
    .paint(featureCollection=all_PAs, color=1)
    .reproject(crs=EE_CRS_1km)
)

# Import covariates

elevation_ic = ee.ImageCollection("COPERNICUS/DEM/GLO30").select("DEM")
slope = elevation_ic.map(lambda tile: ee.Terrain.slope(tile)).mosaic().rename("slope")
elevation = elevation_ic.mosaic().rename("elevation")
treecover2000 = ee.Image("UMD/hansen/global_forest_change_2025_v1_13").select("treecover2000")
travel_time = (
    ee.Image("projects/malariaatlasproject/assets/accessibility/accessibility_to_cities/2015_v1_0")
    .select("accessibility").rename("travel_time")
)
log_pop_density = (
    ee.Image("JRC/GHSL/P2023A/GHS_POP/2000")
    .select("population_count")
    .add(1) # handles zeros for log transform
    .log()
    .rename("log_pop_density")
)

# Resample covariates to 1km resolution

def resample(img):
    return (
        img.setDefaultProjection(EE_CRS_1km)
        .reduceResolution(reducer=ee.Reducer.mean(), maxPixels=4096)
        .reproject(protected_mask.projection())
    )

elevation = resample(elevation)
slope = resample(slope)
treecover2000 = resample(treecover2000)
travel_time = resample(travel_time)
log_pop_density = resample(log_pop_density)

covariates = (
    protected_mask
    .addBands(elevation)
    .addBands(slope)
    .addBands(treecover2000)
    .addBands(travel_time)
    .addBands(log_pop_density)
)

# Apply land mask to avoid sampling oceans and large permanent water bodies

land_mask = (
    ee.Image("UMD/hansen/global_forest_change_2025_v1_13")
    .select("datamask")
    .eq(1)  # 1 = land, 2 = permanent water/ocean, 0 = no data
)
covariates = covariates.updateMask(land_mask)

# Make biome layer and combine it with PA binary to make a strata band
# Each stratum is a unique combo of biome and PA status
# e.g. 6 = unprotected boreal forest, 34 = protected mangroves

biome_fc = ee.FeatureCollection(BIOME_ASSET_ID).map(
    lambda f: f.set("BIOME_NUM", ee.Number(f.get("BIOME_NUM")).int())
)
biome = (
    ee.Image(0)
    .paint(featureCollection=biome_fc, color="BIOME_NUM")
    .rename("biome")
    .reproject(protected_mask.projection())
    .toInt()
)
strata = protected_mask.multiply(20).add(biome).rename("strata").toInt()
covariates = covariates.addBands(strata)

print("Covariate bands:", covariates.bandNames().getInfo())

In [ ]:
# Export the strata image to asset

STRATA_ASSET = f"projects/{PROJECT}/assets/TPAE/strata_1km"

strata_export_task = ee.batch.Export.image.toAsset(
    image=strata,
    description="strata_1km_export",
    assetId=STRATA_ASSET,
    region=ee.Geometry.BBox(-180, -60, 180, 84),
    scale=PSM_CELL_SIZE,
    crs=EE_CRS_METERS,
    maxPixels=1e13,
)
# strata_export_task.start()
# print("Strata export started:", strata_export_task.id)

In [ ]:
# Divide the globe into tiles

tiles_all = build_tile_grid(tile_size_deg=20.0)
# print(f"Total candidate tiles: {len(tiles_all)}")

tiles = filter_tiles_to_land(tiles_all, coarse_scale=10_000, min_land_fraction=0.001)
# print(f"Land tiles: {len(tiles)}")

In [ ]:
# Check that tile grid looks correct

Map = geemap.Map()
Map.add_basemap("CartoDB.Positron")
Map.addLayer(tiles_to_feature_collection(tiles), {"color": "red"}, "Land tiles")
Map.setCenter(0, 20, 2)
Map

In [ ]:
# Globally allocate samples
# Equal number of samples for each biome, with a 2:1 ratio of unprotected to protected.
# Since it is equal allocation, this doesn't require any area calculations, just simple math.

alloc_cache = Path("data/global_allocation.json")

global_allocation = compute_global_allocation(
    total_points=TOTAL_POINTS,
    treat_control_ratio=TREAT_CONTROL,
    min_per_stratum=MIN_PER_STRATUM,
)
# save_allocation(global_allocation, alloc_cache)

In [ ]:
# Check that global allocation worked as expected

alloc_df = pd.DataFrame([
    {"stratum_id": k, "protected": k // 20, "biome": k % 20, "n": v}
    for k, v in global_allocation.items()
]).sort_values("stratum_id")
# print(alloc_df)
# print(f"\nTotal allocated: {alloc_df['n'].sum():,}")
# print(f"Target: {TOTAL_POINTS:,}")

In [ ]:
# Calculate the number of pixels per stratum in each tile

tile_counts_cache = Path("data/tile_pixel_counts.json")

strata_asset = ee.Image(STRATA_ASSET)

tile_pixel_counts = compute_tile_pixel_counts(
    tiles=tiles,
    strata_image=strata_asset,
    scale=PSM_CELL_SIZE,
    projection=ee.Projection(EE_CRS_METERS).atScale(PSM_CELL_SIZE),
    max_workers=10,
)

In [ ]:
# Check that pixel-counting worked as expected

serializable = {k: {str(s): c for s, c in v.items()} for k, v in tile_pixel_counts.items()}
tile_counts_cache.write_text(json.dumps(serializable))
print(f"Computed pixel counts for {len(tile_pixel_counts)} tiles")

# Inspect a few
for tile_id in list(tile_pixel_counts.keys())[:3]:
    counts = tile_pixel_counts[tile_id]
    print(f"\n{tile_id}: {len(counts)} strata, {sum(counts.values()):,} total pixels")

In [ ]:
# Allocate samples to strata within each tile based on pixel counts

tile_allocations = compute_tile_allocations(
    tile_pixel_counts=tile_pixel_counts,
    global_allocation=global_allocation,
    min_tile_stratum=1,
)

In [ ]:
# Check that per-tile sample allocation worked as expected

# Validate: per-stratum sum across tiles should match global budget
validation_df = validate_allocation(tile_allocations, global_allocation, tolerance=0.02)
print(validation_df)

# Per-tile inspection
print("\nPer-tile sample counts (first 10):")
for tile_id in list(tile_allocations.keys())[:10]:
    alloc = tile_allocations[tile_id]
    n_total = sum(alloc.values())
    n_strata = len(alloc)
    print(f"  {tile_id}: {n_total:,} samples across {n_strata} strata")

# Overall
total_samples = sum(sum(a.values()) for a in tile_allocations.values())
print(f"\nGrand total: {total_samples:,}")

Extract covariate values at sample point locations, one tile at a time.
Save samples to GCS bucket.

In [ ]:
# Test on a single tile first

test_tile_id = "10_01"
test_tile_geom = tiles[test_tile_id]
test_alloc = tile_allocations[test_tile_id]

print(f"Test tile: {test_tile_id}")
print(f"  Samples requested: {sum(test_alloc.values()):,}")
print(f"  Strata: {len(test_alloc)}")

test_task, n_req = build_sample_export_task(
    tile_id=test_tile_id,
    tile_geom=test_tile_geom,
    tile_allocation=test_alloc,
    covariates=covariates,
    strata_image=strata_asset,
    sample_bands=["elevation", "slope", "treecover2000", "travel_time", "log_pop_density", "protected"],
    bucket=GCS_BUCKET,
    file_prefix=GCS_PREFIX,
    scale=PSM_CELL_SIZE,
    projection=EE_CRS_1km,
)
test_task.start()
print(f"Task started: {test_task.id}")

import time
for _ in range(60):
    state = test_task.status().get("state")
    print(state)
    if state in {"COMPLETED", "FAILED"}:
        break
    time.sleep(10)
print(test_task.status())

In [ ]:
# Check that the test tile worked as expected

import pandas as pd

test_df = pd.read_csv(f"gs://{GCS_BUCKET}/{GCS_PREFIX}{test_tile_id}.csv")
print(f"Rows: {len(test_df):,}  (requested: {n_req:,})")
print(f"Protected: {test_df['protected'].value_counts().to_dict()}")
print(f"Missing: {test_df.isna().sum().sum()}")
print(f"\nCovariate summaries:")
print(test_df[["elevation", "slope", "treecover2000", "travel_time", "log_pop_density"]].describe())

In [ ]:
# Run the remaining 85 tiles

manager = TileTaskManager(STATE_FILE)
sample_bands = ["elevation", "slope", "treecover2000", "travel_time", "log_pop_density", "protected"]

# Register the test tile as already complete
manager.register(
    tile_id=test_tile_id,
    n_requested=sum(test_alloc.values()),
    gcs_path=f"gs://{GCS_BUCKET}/{GCS_PREFIX}{test_tile_id}.csv",
)
manager.state[test_tile_id].task_id = test_task.id
manager.state[test_tile_id].status = "COMPLETED"
manager.state[test_tile_id].attempts = 1
manager._save()

# Launch all remaining tiles
n_launched = 0
n_skipped = 0
for tile_id, tile_geom in tiles.items():
    alloc = tile_allocations.get(tile_id, {})
    if not alloc:
        n_skipped += 1
        continue

    n_requested = sum(alloc.values())
    gcs_path = f"gs://{GCS_BUCKET}/{GCS_PREFIX}{tile_id}.csv"

    state = manager.register(tile_id, n_requested, gcs_path)
    if state.status == "COMPLETED":
        n_skipped += 1
        continue
    if state.status == "RUNNING":
        n_skipped += 1
        continue

    task, _ = build_sample_export_task(
        tile_id=tile_id,
        tile_geom=tile_geom,
        tile_allocation=alloc,
        covariates=covariates,
        strata_image=strata_asset,
        sample_bands=sample_bands,
        bucket=GCS_BUCKET,
        file_prefix=GCS_PREFIX,
        scale=PSM_CELL_SIZE,
        projection=EE_CRS_1km,
    )
    manager.submit(tile_id, task)
    n_launched += 1

print(f"Launched: {n_launched}")
print(f"Skipped (already complete): {n_skipped}")
print(f"Total registered: {len(manager.state)}")

In [ ]:
# Show failed tile details
failed = [tid for tid, st in manager.state.items() if st.status == "FAILED"]
print(f"\nFailed tiles: {failed}")
for tile_id in failed:
    state = manager.state[tile_id]
    print(f"  {tile_id}: {state.error} (requested: {state.n_requested:,})")

In [ ]:
# Divide failed tiles into smaller sub-tiles and retry

import ee

failed_ids = ["03_05", "05_05", "12_05"]

# Mark parent tiles as split so we don't retry them whole
for pid in failed_ids:
    manager.state[pid].status = "SPLIT"
manager._save()

# For each failed tile, split into 4 quadrants and launch each with 1/4 the allocation
for parent_id in failed_ids:
    parent_geom = tiles[parent_id]
    parent_alloc = tile_allocations[parent_id]
    bounds = parent_geom.bounds().coordinates().get(0).getInfo()
    lon_min, lat_min = bounds[0]
    lon_max, lat_max = bounds[2]
    lon_mid = (lon_min + lon_max) / 2
    lat_mid = (lat_min + lat_max) / 2

    quadrants = {
        f"{parent_id}_sw": [lon_min, lat_min, lon_mid, lat_mid],
        f"{parent_id}_se": [lon_mid, lat_min, lon_max, lat_mid],
        f"{parent_id}_nw": [lon_min, lat_mid, lon_mid, lat_max],
        f"{parent_id}_ne": [lon_mid, lat_mid, lon_max, lat_max],
    }

    # Allocate 1/4 of each stratum's points to each quadrant (good enough approximation)
    sub_alloc = {s: max(1, n // 4) for s, n in parent_alloc.items()}

    for sub_id, coords in quadrants.items():
        sub_geom = ee.Geometry.Rectangle(coords, proj="EPSG:4326", geodesic=False)
        task, _ = build_sample_export_task(
            tile_id=sub_id,
            tile_geom=sub_geom,
            tile_allocation=sub_alloc,
            covariates=covariates,
            strata_image=strata_asset,
            sample_bands=sample_bands,
            bucket=GCS_BUCKET,
            file_prefix=GCS_PREFIX,
            scale=PSM_CELL_SIZE,
            projection=EE_CRS_1km,
        )
        manager.register(sub_id, sum(sub_alloc.values()), f"gs://{GCS_BUCKET}/{GCS_PREFIX}{sub_id}.csv")
        manager.submit(sub_id, task)
        print(f"Launched {sub_id}")

In [ ]:
# Load and concatenate all samples

from google.cloud import storage

client = storage.Client()
bucket = client.bucket(GCS_BUCKET)
blobs = list(client.list_blobs(GCS_BUCKET, prefix=GCS_PREFIX))
print(f"Total CSVs in bucket: {len(blobs)}")

dfs = []
for blob in blobs:
    df = pd.read_csv(f"gs://{GCS_BUCKET}/{blob.name}")
    dfs.append(df)

samples = pd.concat(dfs, ignore_index=True)
print(f"Total samples: {len(samples):,}  (target: 100,000)")
print(f"Protected: {samples['protected'].value_counts().to_dict()}")
print(f"Tiles: {samples['tile_id'].nunique()}")
print(f"\nPer-stratum counts (sorted):")
print(samples["strata"].value_counts().sort_index())

In [ ]:
# Remove samples that ended up in non-biome areas
# (Minor edge effect from RESOLVE dataset)

samples_clean = samples[~samples["strata"].isin([0, 20])].reset_index(drop=True)
print(f"Before: {len(samples):,}")
print(f"After: {len(samples_clean):,}")
print(f"Dropped: {len(samples) - len(samples_clean)}")

In [ ]:
# Thin samples within each stratum so they are at least 3km apart to avoid spatial autocorrelation

# Parse the .geo column into lon, lat columns
samples_clean["lon"] = samples_clean[".geo"].apply(lambda s: json.loads(s)["coordinates"][0])
samples_clean["lat"] = samples_clean[".geo"].apply(lambda s: json.loads(s)["coordinates"][1])

# Project to EPSG:6933 (equal-area meters)
transformer = Transformer.from_crs("EPSG:4326", "EPSG:6933", always_xy=True)
x, y = transformer.transform(samples_clean["lon"].values, samples_clean["lat"].values)
samples_clean["x_m"] = x
samples_clean["y_m"] = y

# Snap to 3 km grid cells
THIN_DISTANCE = 3000
samples_clean["_cell_x"] = (samples_clean["x_m"] // THIN_DISTANCE).astype(np.int64)
samples_clean["_cell_y"] = (samples_clean["y_m"] // THIN_DISTANCE).astype(np.int64)

# Stratify thinning by stratum: keep one sample per (cell_x, cell_y, strata)
# This preserves stratum balance even if a cell has both protected and unprotected samples
samples_thinned = (
    samples_clean
    .sample(frac=1, random_state=42)  # shuffle so duplicate retention is random
    .drop_duplicates(["_cell_x", "_cell_y", "strata"], keep="first")
    .drop(columns=["_cell_x", "_cell_y"])
    .reset_index(drop=True)
)

print(f"Before thinning: {len(samples_clean):,}")
print(f"After thinning:  {len(samples_thinned):,}")
print(f"Removed: {len(samples_clean) - len(samples_thinned):,} ({(1 - len(samples_thinned)/len(samples_clean)):.1%})")

print(f"\nPer-stratum counts after thinning:")
print(samples_thinned["strata"].value_counts().sort_index())

In [ ]:
# Use training data to fit a propensity model.
# This model calculates biome-specific coefficients for each covariate,
# because the effect of each covariate depends on the biome.

COVARIATES = ["elevation", "slope", "treecover2000", "travel_time", "log_pop_density"]

X_raw = samples_thinned[COVARIATES].values
y = samples_thinned["protected"].values
biome = samples_thinned["strata"].values % 20

# Standardize covariates
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# One-hot encode biome (drop first as reference)
biome_dummies = pd.get_dummies(biome, prefix="biome", drop_first=True).astype(float).values

# Build interaction terms: each covariate × each biome dummy
interactions = []
interaction_names = []
biome_levels = sorted(set(biome))[1:]  # skip reference biome
for i, b in enumerate(biome_levels):
    for j, cov in enumerate(COVARIATES):
        interactions.append(X_scaled[:, j] * biome_dummies[:, i])
        interaction_names.append(f"{cov}_x_biome{b}")
interactions = np.column_stack(interactions)

# Combine: main covariates + biome main effects + interactions
X_full = np.hstack([X_scaled, biome_dummies, interactions])
feature_names = (
    COVARIATES
    + [f"biome{b}" for b in biome_levels]
    + interaction_names
)
print(f"Design matrix: {X_full.shape}  ({len(feature_names)} features)")

# Fit
model3 = LogisticRegression(
    penalty=None,
    solver="lbfgs",
    max_iter=5000,
    random_state=42,
)
model3.fit(X_full, y)

# AUC
y_pred_proba = model3.predict_proba(X_full)[:, 1]
auc = roc_auc_score(y, y_pred_proba)
print(f"Training AUC: {auc:.4f}  (Option 1 was 0.6141)")

# Inspect main covariate effects (reference biome only — these are NOT global averages)
main_coefs = pd.DataFrame({
    "covariate": COVARIATES,
    "coef_in_ref_biome": model3.coef_[0][:5],
})
print(f"\nMain covariate effects (in reference biome, biome={sorted(set(biome))[0]}):")
print(main_coefs.to_string(index=False))

# Compute per-biome effective slope for each covariate
# Effective slope in biome b = main_coef + interaction_coef_for_biome_b
print(f"\nPer-biome effective slopes (covariate effect within each biome):")
n_main = 5
n_biome = len(biome_levels)
ref_biome = sorted(set(biome))[0]

# Reference biome row
per_biome_slopes = {ref_biome: model3.coef_[0][:5].tolist()}
for i, b in enumerate(biome_levels):
    # Interaction coefs for this biome occupy interactions[i*5:(i+1)*5]
    interaction_start = n_main + n_biome + i * 5
    interaction_end = interaction_start + 5
    interaction_coefs = model3.coef_[0][interaction_start:interaction_end]
    per_biome_slopes[b] = (model3.coef_[0][:5] + interaction_coefs).tolist()

slope_df = pd.DataFrame(per_biome_slopes, index=COVARIATES).T
slope_df.index.name = "biome"
print(slope_df.round(3))

In [ ]:
# Save the model

import joblib
from pathlib import Path
from datetime import datetime

model_dir = Path("models")
model_dir.mkdir(exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

artifacts = {
    "model": model3,
    "scaler": scaler,
    "covariates": COVARIATES,
    "biome_levels_ref": sorted(set(biome))[0],
    "biome_levels_dummied": biome_levels,
    "feature_names": feature_names,
    "training_metadata": {
        "n_samples": len(y),
        "auc": float(auc),
        "protected_fraction": float(y.mean()),
        "timestamp": timestamp,
        "specification": "Option 3: covariates × biome interactions",
    },
}
joblib.dump(artifacts, model_dir / f"propensity_model_{timestamp}.pkl")
print(f"Saved: models/propensity_model_{timestamp}.pkl")